## Shape ranking per route + direction + headsign

In [0]:
%sql
CREATE OR REPLACE TABLE calgary_transit.gold.route_shape_ranking AS
WITH trips_f AS (
  SELECT route_id, direction_id, trip_headsign, shape_id
  FROM calgary_transit.bronze.gtfs_trips
  WHERE _ingest_date = '2026-02-23'
    AND shape_id IS NOT NULL
),
-- count headsigns inside each (route,dir,shape)
headsign_counts AS (
  SELECT
    route_id,
    direction_id,
    shape_id,
    trip_headsign,
    COUNT(*) AS headsign_trip_count
  FROM trips_f
  GROUP BY route_id, direction_id, shape_id, trip_headsign
),
-- pick the top headsign per (route,dir,shape)
dominant_headsign AS (
  SELECT
    route_id,
    direction_id,
    shape_id,
    trip_headsign AS dominant_headsign
  FROM (
    SELECT
      route_id,
      direction_id,
      shape_id,
      trip_headsign,
      headsign_trip_count,
      ROW_NUMBER() OVER (
        PARTITION BY route_id, direction_id, shape_id
        ORDER BY headsign_trip_count DESC, trip_headsign
      ) AS rn
    FROM headsign_counts
  )
  WHERE rn = 1
),
-- total trips per (route,dir,shape)
shape_counts AS (
  SELECT
    route_id,
    direction_id,
    shape_id,
    COUNT(*) AS trip_count
  FROM trips_f
  GROUP BY route_id, direction_id, shape_id
),
ranked AS (
  SELECT
    sc.route_id,
    sc.direction_id,
    sc.shape_id,
    sc.trip_count,
    dh.dominant_headsign,
    ROW_NUMBER() OVER (
      PARTITION BY sc.route_id, sc.direction_id
      ORDER BY sc.trip_count DESC, sc.shape_id
    ) AS shape_rank,
    SUM(sc.trip_count) OVER (PARTITION BY sc.route_id, sc.direction_id) AS total_trips_dir
  FROM shape_counts sc
  LEFT JOIN dominant_headsign dh
    ON sc.route_id = dh.route_id
   AND sc.direction_id = dh.direction_id
   AND sc.shape_id = dh.shape_id
)
SELECT
  route_id,
  direction_id,
  shape_id,
  dominant_headsign,
  trip_count,
  total_trips_dir,
  ROUND (trip_count / total_trips_dir, 3) AS dominance_ratio,
  shape_rank
FROM ranked;

In [0]:
%sql
select count(*) from calgary_transit.gold.route_shape_ranking ;

In [0]:
%sql
select * from calgary_transit.gold.route_shape_ranking ;

In [0]:
%sql
select count(*)from calgary_transit.gold.route_shape_ranking where shape_rank = 1;